# Notebook 04 — Study-Area Boundaries and Clipping
**Phase:** prepare the three-mandal study boundary and clip the masked NDVI raster.

Study area: Anantapur Urban, Anantapur Rural, and Bukkaraya Samudram. The project summary reports EPSG:32643 (WGS 84 / UTM zone 43N) for metric analysis.

In [ ]:
%pip install geopandas rasterio shapely pyproj matplotlib

In [ ]:
from pathlib import Path
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
import matplotlib.pyplot as plt

BOUNDARY_PATH = Path(r"data/processed/Three_Mandals_Study_Area.geojson")
NDVI_PATH = Path(r"data/processed/Anantapur_NDVI_Masked_20261009.tif")
OUTPUT_PATH = Path(r"data/processed/Three_Mandals_NDVI_Clipped_20261009.tif")
print("Boundary exists:", BOUNDARY_PATH.exists(), "NDVI exists:", NDVI_PATH.exists())

## 1. Inspect the boundary and CRS
Do not guess a missing CRS. Identify it from the source metadata before reprojection.

In [ ]:
if BOUNDARY_PATH.exists():
    boundary = gpd.read_file(BOUNDARY_PATH)
    print("Features:", len(boundary), "CRS:", boundary.crs)
    print("Geometry types:", boundary.geom_type.value_counts().to_dict())
    if boundary.crs is None:
        raise ValueError("Boundary CRS missing; identify source CRS before proceeding.")
    display(boundary.head())
else:
    print("Correct BOUNDARY_PATH before continuing.")

## 2. Clip NDVI to the final boundary

In [ ]:
if BOUNDARY_PATH.exists() and NDVI_PATH.exists():
    boundary = gpd.read_file(BOUNDARY_PATH)
    if boundary.crs is None:
        raise ValueError("Boundary CRS missing.")
    with rasterio.open(NDVI_PATH) as src:
        projected = boundary.to_crs(src.crs)
        geoms = [g.__geo_interface__ for g in projected.geometry if g is not None]
        nodata = src.nodata if src.nodata is not None else -9999.0
        clipped, transform = mask(src, geoms, crop=True, filled=True, nodata=nodata)
        profile = src.profile.copy()
        profile.update(height=clipped.shape[1], width=clipped.shape[2],
                       transform=transform, nodata=nodata, compress="deflate")
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(OUTPUT_PATH, "w", **profile) as dst: dst.write(clipped)
    print("Saved:", OUTPUT_PATH)
else:
    print("Skipped — set valid boundary and NDVI paths.")

## 3. Do we need the district boundary?
The district boundary and `Anantapur_NDVI_District_Clipped_20261009.tif` are intermediate/optional if the final analysis is only for the three mandals. Keep them if a script or QGIS project references them or if you need to reproduce the district-wide intermediate step. The three-mandal boundary is more directly relevant to the final study area.

Before deleting any local files, search scripts and QGIS layer sources for their filenames.

## Troubleshooting
- **CRS missing:** verify the source CRS; do not assign one arbitrarily.
- **Empty clip/no overlap:** check CRS, geometry validity, and raster extent.
- **Unexpected file size:** cropping, compression, and pixel count determine output size.
- **QGIS layers broken after moving files:** repair data sources or use documented relative paths.

## Phase outcome
Validate the clipped raster's CRS, resolution, dimensions, NoData, and visual alignment against the three-mandal boundary.